# Kuramoto-Sivashinsky Equation Energy-Preserving Operator Inference

This notebook trains reduced-order models (ROMs) of the chaotic Kuramoto–Sivashinsky equation (KSE)

$$
\begin{gather*}
    \frac{\partial u}{\partial t} = -\mu\frac{\partial^4 u}{\partial \omega^4} - \frac{\partial^2 u}{\partial \omega^2} - u\frac{\partial u}{\partial \omega}, \qquad \omega \in [0, 22), \quad \text{periodic BCs},
\end{gather*}
$$

with standard (least-squares) Operator Inference (OpInf), the intrusive POD-Galerkin projection, and the three energy-preserving OpInf (EP-OpInf) variants of LiftAndLearn: hard equality constraints (EPHEC), soft inequality constraints (EPSIC) and an energy-preserving penalty (EPP). The models are then compared on the training data: projection error, state error, energy-preserving constraint residual, autocorrelation, Lyapunov exponents and Kaplan–Yorke dimension.

**Demo vs. paper scale.** By default the notebook runs a reduced *demo* setup in about 5–15 minutes (single-threaded; it also uses `Threads.@threads` if Julia was started with several threads). Set `PAPER_SCALE = true` in the settings cell to reproduce the full experiment from the paper. That needs several hours and a lot of memory (≳64 GB), because the full-order trajectories are then kept at every time step.

**Files written for the test notebook `07_KSE_EP-OpInf_test.ipynb`** (in `data/`):

| file | contents |
|---|---|
| `data/kse_data_L22.jld2` | `"settings"` (all simulation settings, incl. `PAPER_SCALE`), POD basis `"Vr"`, singular values `"Σr"`, reduced orders `"ro"`, full-order operators `"op_fom_tr"`, training ICs `"IC_train"`, training data `"Xtr"`, `"Rtr"`, `"Xtr_all"` |
| `data/kse_operators_L22.jld2` | ROM operators `"op_LS"`, `"op_int"`, `"op_ephec"`, `"op_epsic"`, `"op_epp"` (all of dimension `ro[end]`) |
| `data/kse_results_L22.jld2` | training results (`RES` dictionary) |

Run this notebook before notebook 07.

In [ ]:
using DataFrames
using FileIO
using JLD2
using LaTeXStrings
using LinearAlgebra
using Plots
using Plots.PlotMeasures
using ProgressMeter
using Random
using SparseArrays
using Statistics
using StatsBase
using UniqueKronecker
import HSL_jll
using PolynomialModelReductionDataset: KuramotoSivashinskyModel

using LiftAndLearn
const LnL = LiftAndLearn
;

## Settings

In [ ]:
# Set to `true` to reproduce the full-size experiments from the paper (takes hours);
# the default demo settings run in a few minutes.
const PAPER_SCALE = false

# Simulation settings (identical in both modes)
Ω = (0.0, 22.0)     # spatial domain (L = 22)
Nx = 2^9            # number of grid points
dt = 1e-3           # time step
T_END = 300.0       # final time of every simulation

# Downsampling of the training snapshots used for POD and operator inference (every DS-th time step).
# The demo uses fewer snapshots so that the EP-OpInf optimization problems stay small.
DS = PAPER_SCALE ? 100 : 1000

# Snapshots every EVAL_STRIDE time steps (0.1 time units) are used for the state errors, the
# autocorrelation lags and the flow-field plots.
EVAL_STRIDE = 100

# Stride (in time steps) at which the full-order trajectories are kept in memory for the analysis.
# The paper keeps every time step (this is what needs the large memory).
SAVE_STRIDE = PAPER_SCALE ? 1 : EVAL_STRIDE

# Reduced orders: the POD truncation levels with relative energy loss 10^-2, 10^-3, ... (see below).
# The demo stops at a smaller maximum order r_max.
RO_LEVELS = PAPER_SCALE ? (2:8) : (2:5)

# Maximum number of Ipopt iterations of the EP-OpInf optimizations
MAX_ITER = 1000

# Lyapunov exponent computation: m exponents, spin-up time τ, re-orthonormalization every T time units,
# integration time step Δt, N re-orthonormalizations, perturbation size ϵ
LE_OPTS = PAPER_SCALE ?
    (m=10, τ=2e3, T=1.0, Δt=0.01, N=10_000, ϵ=1e-6) :
    (m=10, τ=1e2, T=1.0, Δt=0.01, N=1_000, ϵ=1e-6)

# Files
mkpath("data")
mkpath("figures")
datafile = "data/kse_data_L22.jld2"
opfile = "data/kse_operators_L22.jld2"
resultfile = "data/kse_results_L22.jld2"
;

In [ ]:
# KSE model (finite differences with energy-preserving discretization of the nonlinear term)
KSE = KuramotoSivashinskyModel(
    spatial_domain=Ω, time_domain=(0.0, T_END), Δx=(Ω[2] - Ω[1]) / Nx, Δt=dt,
    diffusion_coeffs=1.0, BC=:periodic, conservation_type=:EP,
)

# Use HSL's ma97 if a licensed HSL library is installed; otherwise Ipopt's default (MUMPS)
const USE_HSL = ccall((:LIBHSL_isfunctional, HSL_jll.libhsl), Bool, ())

# Settings for Operator Inference
KSE_system = LnL.SystemStructure(
    state=[1, 2],  # linear + quadratic state operators
)
KSE_vars = LnL.VariableStructure(
    N=1,
)
KSE_data = LnL.DataStructure(
    Δt=KSE.Δt,
    DS=DS,
)
KSE_optim = LnL.OptimizationSetting(
    verbose=false,  # set to true to print Ipopt's iteration log
    initial_guess=false,
    max_iter=MAX_ITER,
    reproject=false,
    SIGE=false,
    with_bnds=true,
    linear_solver = USE_HSL ? "ma97" : "none",
    HSL_lib_path = USE_HSL ? HSL_jll.libhsl_path : "none",
)

# Number of time steps of each simulation
nsteps = KSE.time_dim - 1
println("spatial dimension = $(KSE.spatial_dim), time steps = $(nsteps), HSL ma97 available: $(USE_HSL)")
;

### A fast integrator for the full-order model

`KSE.integrate_model` integrates $\dot{\mathbf{x}} = \mathbf{A}\mathbf{x} + \mathbf{F}(\mathbf{x}\oslash\mathbf{x})$ with a Crank–Nicolson/Adams–Bashforth (CNAB) scheme. At every step it forms the full unique Kronecker product $\mathbf{x}\oslash\mathbf{x}$, which has $n(n+1)/2 = 131{,}328$ entries for $n = 512$. That makes long full-order simulations slow (about 1 ms per step). The full-order operator $\mathbf{F}$ has only $4n$ nonzeros, so the small stepper below runs the *same* CNAB scheme but evaluates the quadratic term from the nonzeros of $\mathbf{F}$ only, and uses a sparse LU factorization for the implicit part. It is about 100× faster for the full-order model. We check below that it reproduces `KSE.integrate_model`.

The ROMs are simulated with `KSE.integrate_model`. The CNAB stepper is also used for the Lyapunov-exponent computation, where we need many short simulations.

In [ ]:
"""
    CNABStepper(A, F, Δt)

Crank–Nicolson (linear part) / Adams–Bashforth (quadratic part) time stepper for
`ẋ = A x + F (x ⊘ x)`, i.e. the scheme of `KSE.integrate_model`, evaluating `F (x ⊘ x)`
from the nonzeros of `F`.
"""
struct CNABStepper
    Δt::Float64
    Mfac                  # factorization of I - Δt/2 A
    P                     # I + Δt/2 A
    qrow::Vector{Int}     # nonzeros of F: F[qrow, col] multiplies x[qi] * x[qj]
    qi::Vector{Int}
    qj::Vector{Int}
    qval::Vector{Float64}
end

function CNABStepper(A, F, Δt)
    n = size(A, 1)
    pairs = [(i, j) for i in 1:n for j in i:n]  # column ordering of x ⊘ x
    rows, cols, vals = findnz(sparse(F))
    if issparse(A)
        Mfac = lu(sparse(1.0I(n) - Δt/2 * A))
        P = sparse(1.0I(n) + Δt/2 * A)
    else
        Mfac = lu(Matrix(1.0I(n) - Δt/2 * A))
        P = Matrix(1.0I(n) + Δt/2 * A)
    end
    return CNABStepper(Δt, Mfac, P, rows, first.(pairs[cols]), last.(pairs[cols]), vals)
end

function quad!(f, S::CNABStepper, u)
    fill!(f, 0.0)
    @inbounds for p in eachindex(S.qval)
        f[S.qrow[p]] += S.qval[p] * u[S.qi[p]] * u[S.qj[p]]
    end
    return f
end

"""
    cnab_solve(S, u0, nsteps; keep=0:nsteps)

Integrate `nsteps` time steps from `u0` and return the states at the time steps listed in the
sorted vector `keep` (step 0 is the initial condition) as the columns of a matrix.
"""
function cnab_solve(S::CNABStepper, u0::AbstractVector, nsteps::Integer; keep=0:nsteps)
    @assert issorted(keep) && first(keep) >= 0 && last(keep) <= nsteps
    X = zeros(length(u0), length(keep))
    u = Vector{Float64}(u0)
    f = zero(u); fold = zero(u)
    c = 1
    while c <= length(keep) && keep[c] == 0
        X[:, c] = u; c += 1
    end
    for k in 1:nsteps
        quad!(f, S, u)
        if k == 1  # forward Euler for the quadratic term in the first step (as in KSE.integrate_model)
            rhs = S.P * u .+ S.Δt .* f
        else
            rhs = S.P * u .+ (1.5 * S.Δt) .* f .- (0.5 * S.Δt) .* fold
        end
        u = S.Mfac \ rhs
        f, fold = fold, f
        while c <= length(keep) && keep[c] == k
            X[:, c] = u; c += 1
        end
    end
    return X
end
;

In [ ]:
# Check against the library integrator on a short time interval
A_chk, F_chk = KSE.finite_diff_model(KSE, KSE.diffusion_coeffs[1])
u_chk = cos.(2π * KSE.xspan / 22) .+ 0.4 * cos.(4π * KSE.xspan / 22)
X_lib = KSE.integrate_model(KSE.tspan[1:2001], u_chk; linear_matrix=A_chk, quadratic_matrix=F_chk,
                            system_input=false, const_stepsize=true)
X_fast = cnab_solve(CNABStepper(A_chk, F_chk, KSE.Δt), u_chk, 2000)
println("relative difference to KSE.integrate_model over t ∈ [0, 2]: ", norm(X_lib - X_fast) / norm(X_lib))
;

## Generate Training Data

We use the initial condition of

$$
\begin{gather*}
    u(\omega,0) = a\cos\left( \frac{2\pi \omega}{L} \right) + b\cos\left( \frac{4\pi \omega}{L} \right)
\end{gather*}
$$

where $a$ and $b$ are parameters that vary.

In [ ]:
# Parameters of the initial condition
ic_a = [0.8, 1.0, 1.2]
ic_b = [0.2, 0.4, 0.6]

num_ic_params = Int(length(ic_a) * length(ic_b))
L = KSE.spatial_domain[2] - KSE.spatial_domain[1]  # length of the domain

# Parameterized function for the initial condition
u0 = (a,b) -> a * cos.((2*π*KSE.xspan)/L) .+ b * cos.((4*π*KSE.xspan)/L)  # initial condition
;

In [ ]:
# Store values
Xtr = Vector{Matrix{Float64}}(undef, KSE.param_dim)  # training state data
Rtr = Vector{Matrix{Float64}}(undef, KSE.param_dim)  # training derivative data
Xtr_all = Matrix{Matrix{Float64}}(undef, KSE.param_dim, num_ic_params)  # FOM trajectories (every SAVE_STRIDE steps)
IC_train = Vector{Vector{Float64}}(undef, num_ic_params)  # all initial conditions
Vr = Vector{Matrix{Float64}}(undef, KSE.param_dim)  # POD basis
Σr = Vector{Vector{Float64}}(undef, KSE.param_dim)  # singular values
op_fom_tr = Vector{LnL.Operators}(undef, KSE.param_dim)  # FOM operators

# Time steps to keep from each simulation
k_snap = 1:DS:nsteps          # training snapshots (time steps 1, 1+DS, 1+2DS, ...)
k_save = 0:SAVE_STRIDE:nsteps  # trajectory kept for the analysis
keep = sort(unique(vcat(k_save, k_snap .- 1, k_snap)))
col = k -> searchsortedfirst(keep, k)  # column of time step k in the solution

ic_combos = vec(collect(Iterators.product(ic_a, ic_b)))

@info "Generate the FOM system matrices and training data."
for i in eachindex(KSE.diffusion_coeffs)
    μ = KSE.diffusion_coeffs[i]

    # Generate the FOM system matrices (ONLY DEPENDS ON μ)
    A, F = KSE.finite_diff_model(KSE, μ)
    op_fom_tr[i] = LnL.Operators(A=A, A2u=F)

    # Store the training data
    Xall = Vector{Matrix{Float64}}(undef, num_ic_params)
    Xdotall = Vector{Matrix{Float64}}(undef, num_ic_params)

    # Generate the data for all combinations of the initial condition parameters
    prog = Progress(length(ic_combos))
    Threads.@threads for j in eachindex(ic_combos)
        a, b = ic_combos[j]
        S = CNABStepper(A, F, KSE.Δt)  # one stepper per task (the LU factorization is not thread-safe)
        states = cnab_solve(S, u0(a, b), nsteps; keep=keep)

        Xtr_all[i, j] = states[:, col.(k_save)]
        Xall[j] = states[:, col.(k_snap)]  # downsampled states
        Xdotall[j] = (states[:, col.(k_snap)] - states[:, col.(k_snap .- 1)]) / KSE.Δt  # backward-difference derivatives
        if i == 1
            IC_train[j] = u0(a, b)
        end
        next!(prog)
    end
    # Combine all initial condition data to form one big training data matrix
    Xtr[i] = reduce(hcat, Xall)
    Rtr[i] = reduce(hcat, Xdotall)

    # Compute the POD basis from the training data
    tmp = svd(Xtr[i])
    Vr[i] = tmp.U
    Σr[i] = tmp.S
end
println("training data: ", size(Xtr[1], 2), " snapshots of dimension ", size(Xtr[1], 1))

### Check Energy-Levels from SVD and Decide Reduced Orders

In [ ]:
nice_orders_all = Vector{Vector{Int}}(undef, KSE.param_dim)
energy_level_all = Vector{Vector{Float64}}(undef, KSE.param_dim)

for i in eachindex(KSE.diffusion_coeffs)
    nice_orders_all[i], energy_level_all[i] = LnL.choose_ro(Σr[i]; en_low=-12)
end
nice_orders = Int.(round.(mean(nice_orders_all)))
energy_level = mean(energy_level_all)
;

In [ ]:
rmax_level = last(RO_LEVELS)
plot(energy_level[1:nice_orders[end]], yaxis=:log10, label="energy loss", fontfamily="Computer Modern", lw=2,
    ylabel="relative energy loss from truncation", xlabel="number of retained modes", legend=:topright, majorgrid=true, grid=true)
plot!(nice_orders, energy_level[nice_orders], seriestype=:scatter, label=nothing, markersize=6)
annotate!((nice_orders[rmax_level]+4, energy_level[nice_orders[rmax_level]], text(L"r_{\mathrm{max}}="*string(nice_orders[rmax_level]), :bottom, 18, "Computer Modern")))
yticks!(10.0 .^ (0:-2:-10))
plot!(fg_legend=:false)
plot!(guidefontsize=14, tickfontsize=12, titlefontsize=15,  legendfontsize=15, right_margin=2mm)

**Based on the energy levels we select the reduced orders.** `ro` contains the orders at which the relative energy loss first drops below $10^{-2}, 10^{-3}, \dots$ (up to $10^{-8}$ in the paper, $10^{-5}$ in the demo). All ROMs are learned at $r_{\max}$ = `ro[end]`; the smaller ROMs are obtained by truncating the operators.

In [ ]:
ro = nice_orders[RO_LEVELS]
println("reduced orders: ", ro)
;

### Save data

In [ ]:
SETTINGS = Dict(
    "PAPER_SCALE" => PAPER_SCALE, "Omega" => Ω, "Nx" => Nx, "dt" => dt, "T_END" => T_END,
    "mu" => KSE.diffusion_coeffs, "DS" => DS, "EVAL_STRIDE" => EVAL_STRIDE, "SAVE_STRIDE" => SAVE_STRIDE,
    "ic_a" => ic_a, "ic_b" => ic_b,
)
DATA = Dict(
    "settings" => SETTINGS,
    "Xtr" => Xtr, "Rtr" => Rtr, "Vr" => Vr,
    "op_fom_tr" => op_fom_tr, "Σr" => Σr,
    "Xtr_all" => Xtr_all, "IC_train" => IC_train,
    "ro" => ro,
)
save(datafile, DATA)

## Obtain Operators using Least-Squares (Vanilla) Operator Inference

In [ ]:
@info "Compute the Least Squares solution."
options = LnL.LSOpInfOption(
    system=KSE_system,
    vars=KSE_vars,
    data=KSE_data,
    optim=KSE_optim,
)

# Store values
op_LS = Array{LnL.Operators}(undef, KSE.param_dim)

@showprogress for i in eachindex(KSE.diffusion_coeffs)
    op_LS[i] = LnL.opinf(Xtr[i], Vr[i][:, 1:ro[end]], options; Xdot=Rtr[i])
end

## Compute intrusive operators

In [ ]:
@info "Compute the intrusive model"

# Store values
op_int = Array{LnL.Operators}(undef, KSE.param_dim)

for i in eachindex(KSE.diffusion_coeffs)
    # Compute the values for the intrusive model from the basis of the training data
    op_int[i] = LnL.pod(op_fom_tr[i], Vr[i][:, 1:ro[end]], KSE_system)
end

## Compute EPHEC-OpInf operators

The EP-OpInf problems are solved with JuMP + Ipopt. After each solve, LiftAndLearn logs a summary that includes Ipopt's termination status (`LOCALLY_SOLVED` is expected). Set `verbose=true` in `KSE_optim` to also see Ipopt's iteration log.

In [ ]:
@info "Compute the EPHEC model"

options = LnL.EPHECOpInfOption(
    system=KSE_system,
    vars=KSE_vars,
    data=KSE_data,
    optim=KSE_optim,
    linear_operator_bounds=(-1000.0, 1000.0),
    quad_operator_bounds=(-100.0, 100.0),
)
op_ephec =  Array{LnL.Operators}(undef, KSE.param_dim)

for i in eachindex(KSE.diffusion_coeffs)
    op_ephec[i] = LnL.epopinf(Xtr[i], Vr[i][:, 1:ro[end]], options; Xdot=Rtr[i])
end
println("EPHEC constraint residual at r = $(ro[end]): ", LnL.ep_constraint_residual(op_ephec[1].A2u, ro[end]))

## Compute EPSIC-Opinf operators

In [ ]:
@info "Compute the EPSIC OpInf."

options = LnL.EPSICOpInfOption(
    system=KSE_system,
    vars=KSE_vars,
    data=KSE_data,
    optim=KSE_optim,
    ϵ=1e-3,
    linear_operator_bounds=(-1000.0, 1000.0),
    quad_operator_bounds=(-100.0, 100.0),
)
op_epsic = Array{LnL.Operators}(undef, KSE.param_dim)

for i in eachindex(KSE.diffusion_coeffs)
    op_epsic[i] = LnL.epopinf(Xtr[i], Vr[i][:, 1:ro[end]], options; Xdot=Rtr[i])
end
println("EPSIC constraint residual at r = $(ro[end]): ", LnL.ep_constraint_residual(op_epsic[1].A2u, ro[end]))

## Compute EPP-OpInf operators

In [ ]:
@info "Compute the EPP OpInf."

options = LnL.EPPOpInfOption(
    system=KSE_system,
    vars=KSE_vars,
    data=KSE_data,
    optim=KSE_optim,
    α=1e6,
    linear_operator_bounds=(-1000.0, 1000.0),
    quad_operator_bounds=(-100.0, 100.0),
)
op_epp =  Array{LnL.Operators}(undef, KSE.param_dim)

for i in eachindex(KSE.diffusion_coeffs)
    op_epp[i] = LnL.epopinf(Xtr[i], Vr[i][:, 1:ro[end]], options; Xdot=Rtr[i])
end
println("EPP constraint residual at r = $(ro[end]): ", LnL.ep_constraint_residual(op_epp[1].A2u, ro[end]))

## Save the operators

In [ ]:
OPS = Dict(
    "op_LS" => op_LS, "op_int" => op_int, "op_ephec" => op_ephec,
    "op_epsic" => op_epsic, "op_epp" => op_epp,
)
save(opfile, OPS)

## Analyze Training Results

The analysis below reads everything from the dictionaries `DATA` and `OPS`, i.e. the objects that were just saved. After a kernel restart you can skip the training and continue from here by first running the setup cells (packages, settings and the CNAB stepper) and then `DATA = load(datafile); OPS = load(opfile); ro = DATA["ro"]`.

All ROMs below are simulated with `KSE.integrate_model`. The helper truncates the $r_{\max}$-dimensional operators to dimension $r$.

In [ ]:
"""Simulate the r-dimensional ROM obtained by truncating the operators `op` from the initial condition `x0`."""
function simulate_rom(model, op, r, x0; tspan=model.tspan)
    Ar = op.A[1:r, 1:r]
    Fr = UniqueKronecker.extractF(op.A2u, r)
    return model.integrate_model(tspan, x0; linear_matrix=Ar, quadratic_matrix=Fr,
                                 system_input=false, const_stepsize=true)
end

# Keys and labels of the ROMs
rom_keys = (:LS, :int, :ephec, :epsic, :epp)
rom_ops = Dict(:LS => OPS["op_LS"], :int => OPS["op_int"], :ephec => OPS["op_ephec"],
               :epsic => OPS["op_epsic"], :epp => OPS["op_epp"])

# Results
RES = Dict{String, Any}()
;

### Projection error

In [ ]:
function analyze_proj_err(model, X_all, Vr_all, ro)
    PE_all = Array{Float64}(undef, length(ro), model.param_dim)
    for i in eachindex(model.diffusion_coeffs), (j, r) in enumerate(ro)
        Vr = Vr_all[i][:, 1:r]
        PE_all[j, i] = mean(LnL.proj_error(X, Vr) for X in X_all[i, :])
    end
    return PE_all
end
;

In [ ]:
RES["train_proj_err"] = analyze_proj_err(KSE, DATA["Xtr_all"], DATA["Vr"], DATA["ro"]);

In [ ]:
# Training data
mean_train_proj_err = mean(RES["train_proj_err"], dims=2)
plot(DATA["ro"], mean_train_proj_err, marker=(:rect), fontfamily="Computer Modern")
plot!(yscale=:log10, majorgrid=true, legend=false)
xticks!(DATA["ro"])
xlabel!("reduced model dimension " * L"r")
ylabel!("projection error")
plot!(guidefontsize=16, tickfontsize=13,  legendfontsize=13)

### Relative state error

The KSE is chaotic: every ROM trajectory eventually decorrelates from the full-order trajectory, so the state error over the whole time horizon is $\mathcal{O}(1)$ for *all* models. We therefore also report the error over the short horizon $t \in [0, T_{\text{short}}]$ with $T_{\text{short}} = 20$. The statistics below (autocorrelation, Lyapunov exponents) are the more meaningful comparison for long times.

In [ ]:
T_SHORT = 20.0                          # short horizon (time units)
k_short = round(Int, T_SHORT / KSE.Δt)  # number of time steps of the short horizon

function analyze_rse(op, model, X_all, Vr_all, IC, ro)
    # Relative state error over the full and the short horizon (snapshots every EVAL_STRIDE time steps)
    RSE_all = Array{Float64}(undef, length(ro), model.param_dim)
    RSE_short_all = Array{Float64}(undef, length(ro), model.param_dim)
    RSE = Array{Float64}(undef, length(IC))
    RSE_short = Array{Float64}(undef, length(IC))
    s = EVAL_STRIDE ÷ SAVE_STRIDE  # stride within the stored FOM trajectories

    for i in eachindex(model.diffusion_coeffs)
        prog = Progress(length(ro))
        for (j, r) in enumerate(ro)
            Vr = Vr_all[i][:, 1:r]
            Threads.@threads for ct in eachindex(IC)
                X = simulate_rom(model, op[i], r, Vr' * IC[ct])
                RSE[ct] = LnL.rel_state_error(X_all[i, ct][:, 1:s:end], X[:, 1:EVAL_STRIDE:end], Vr)
                RSE_short[ct] = LnL.rel_state_error(X_all[i, ct][:, 1:s:(k_short ÷ SAVE_STRIDE + 1)],
                                                    X[:, 1:EVAL_STRIDE:k_short+1], Vr)
            end
            RSE_all[j, i] = mean(RSE)
            RSE_short_all[j, i] = mean(RSE_short)
            next!(prog)
        end
    end
    return RSE_all, RSE_short_all
end
;

In [ ]:
RES["train_state_err"] = Dict{Symbol, Matrix{Float64}}()
RES["train_state_err_short"] = Dict{Symbol, Matrix{Float64}}()
for key in rom_keys
    RES["train_state_err"][key], RES["train_state_err_short"][key] =
        analyze_rse(rom_ops[key], KSE, DATA["Xtr_all"], DATA["Vr"], DATA["IC_train"], DATA["ro"])
end
println("Average relative state error over t ∈ [0, $(T_END)]:")
show(stdout, MIME("text/plain"), DataFrame(r=DATA["ro"]; (key => vec(RES["train_state_err"][key]) for key in rom_keys)...))
println("\n\nAverage relative state error over t ∈ [0, $(T_SHORT)]:")
DataFrame(r=DATA["ro"]; (key => vec(RES["train_state_err_short"][key]) for key in rom_keys)...)

In [ ]:
plot(DATA["ro"], RES["train_state_err"][:LS], c=:black, marker=(:circle, 3.5, :black), label="opinf")
plot!(DATA["ro"], RES["train_state_err"][:int], c=:orange, marker=(:cross, 8, :orange), label="intrusive")
plot!(DATA["ro"], RES["train_state_err"][:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label="ephec-opinf")
plot!(DATA["ro"], RES["train_state_err"][:epsic], c=:purple, markerstrokecolor=:purple, marker=(:dtriangle, 5), ls=:dot, label="epsic-opinf")
plot!(DATA["ro"], RES["train_state_err"][:epp], c=:red, markerstrokecolor=:red, marker=(:star, 4), lw=1, ls=:dash, label="epp-opinf")
plot!(majorgrid=true, legend=:topright)
xticks!(DATA["ro"])
xlabel!("reduced model dimension " * L" r")
ylabel!("average relative state error")
title!("full horizon " * L"t \in [0, %$(Int(T_END))]", titlefontsize=14)
p_full = plot!(guidefontsize=14, tickfontsize=12, legendfontsize=11, fontfamily="Computer Modern")

p_short = plot(DATA["ro"], RES["train_state_err_short"][:LS], c=:black, marker=(:circle, 3.5, :black), label="opinf")
plot!(DATA["ro"], RES["train_state_err_short"][:int], c=:orange, marker=(:cross, 8, :orange), label="intrusive")
plot!(DATA["ro"], RES["train_state_err_short"][:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label="ephec-opinf")
plot!(DATA["ro"], RES["train_state_err_short"][:epsic], c=:purple, markerstrokecolor=:purple, marker=(:dtriangle, 5), ls=:dot, label="epsic-opinf")
plot!(DATA["ro"], RES["train_state_err_short"][:epp], c=:red, markerstrokecolor=:red, marker=(:star, 4), lw=1, ls=:dash, label="epp-opinf")
plot!(majorgrid=true, legend=:topright, yscale=:log10)
xticks!(DATA["ro"])
xlabel!("reduced model dimension " * L" r")
title!("short horizon " * L"t \in [0, %$(Int(T_SHORT))]", titlefontsize=14)
plot!(guidefontsize=14, tickfontsize=12, legendfontsize=11, fontfamily="Computer Modern")

plot(p_full, p_short, layout=(1, 2), size=(1200, 450), bottom_margin=6mm, left_margin=6mm)

### Constraint Residual

The energy-preserving constraint residual $\sum_{i,j,k}|\hat{h}_{ijk} + \hat{h}_{jik} + \hat{h}_{kji}|$ of the quadratic operator (`LnL.ep_constraint_residual`). The full-order operator and the intrusive ROM are energy preserving by construction. EPHEC enforces the constraint exactly, EPSIC up to $\epsilon$ per constraint, and EPP penalizes it.

In [ ]:
function analyze_cr(op, model, ro)
    CR_all = Array{Float64}(undef, length(ro), model.param_dim)
    for i in eachindex(model.diffusion_coeffs), (j, r) in enumerate(ro)
        CR_all[j, i] = LnL.ep_constraint_residual(UniqueKronecker.extractF(op[i].A2u, r), r)
    end
    return CR_all
end

function fom_CR(op, model)
    CR = Array{Float64}(undef, model.param_dim)
    for i in eachindex(model.diffusion_coeffs)
        F_full = op[i].A2u
        CR[i] = LnL.ep_constraint_residual(F_full, size(F_full, 1))
    end
    return CR
end
;

In [ ]:
RES["train_CR"] = Dict{Symbol, Array{Float64}}()

# Compute CR of full order model
RES["train_CR"][:fom] = fom_CR(DATA["op_fom_tr"], KSE)
println("FOM constraint residual: ", RES["train_CR"][:fom])

# ROMs
for key in rom_keys
    RES["train_CR"][key] = analyze_cr(rom_ops[key], KSE, DATA["ro"])
end
DataFrame(r=DATA["ro"]; (key => vec(RES["train_CR"][key]) for key in rom_keys)...)

In [ ]:
# Zero residuals are clipped to 1e-16 for the log scale
clip(x) = max.(x, 1e-16)
plot(DATA["ro"], clip(RES["train_CR"][:int]), c=:orange, marker=(:cross, 10), markerstrokewidth=2.5, label="Intrusive")
plot!(DATA["ro"], clip(RES["train_CR"][:LS]), marker=(:circle, 5), c=:crimson, markerstrokecolor=:crimson, lw=2, label="OpInf")
plot!(DATA["ro"], clip(RES["train_CR"][:ephec]), c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), lw=2, ls=:dash, label="EPHEC-OpInf")
plot!(DATA["ro"], clip(RES["train_CR"][:epsic]), c=:purple, markerstrokecolor=:purple, marker=(:dtriangle, 5), ls=:dot, label="EPSIC-OpInf")
plot!(DATA["ro"], clip(RES["train_CR"][:epp]), c=:red, markerstrokecolor=:red, marker=(:star, 4), ls=:dash, lw=1, label="EPP-OpInf")
plot!(yscale=:log10, majorgrid=true, legend=:right, minorgridalpha=0.03)
yticks!(10.0 .^ [-15, -12, -9, -6, -3, 0, 3])
xticks!(DATA["ro"])
xlabel!("reduced model dimension " * L" r")
ylabel!("energy-preserving constraint residual")
plot!(xlabelfontsize=16, ylabelfontsize=13, tickfontsize=13, legendfontsize=12, fontfamily="Computer Modern")

### Autocorrelation

The normalized autocorrelation $\mathrm{autocor}(u(\omega_i,\cdot))$ is computed at every grid point and averaged over the domain. FOM statistics come from the stored training trajectories (every `SAVE_STRIDE` steps), and the ROM statistics from ROM simulations started at the same initial conditions. The lags are spaced `EVAL_STRIDE` time steps apart up to half the time horizon. In the demo the stored trajectories are sampled every `EVAL_STRIDE` steps, so the autocorrelation is estimated from those samples; the paper uses every time step.

In [ ]:
# Mean (over the grid points) normalized autocorrelation of a trajectory X (columns = time samples)
function tmean_autocorr(X::AbstractArray, lags::AbstractVector)
    N = size(X, 1)
    Cx = zeros(N, length(lags))
    for i in 1:N  # normalized autocorrelation
        Cx[i, :] = autocor(X[i, :], lags)
    end
    return vec(mean(Cx, dims=1))
end

# Autocorrelation of the ROMs of all orders `ro` started from the full-order initial condition `IC`
# (first parameter value only); returns a (lags × ro) matrix
function analyze_autocorr(op, model, Vr_all, IC, ro, lags)
    ac = Matrix{Float64}(undef, length(lags), length(ro))
    for (j, r) in enumerate(ro)
        Vr = Vr_all[1][:, 1:r]
        X = simulate_rom(model, op[1], r, Vr' * IC)
        ac[:, j] = tmean_autocorr(Vr * X[:, 1:SAVE_STRIDE:end], lags)
    end
    return ac
end
;

In [ ]:
# Time lags in units of the stored samples (spaced EVAL_STRIDE time steps apart, up to T_END/2)
lags = 0:(EVAL_STRIDE ÷ SAVE_STRIDE):((KSE.time_dim ÷ 2) ÷ SAVE_STRIDE)
lags_t = collect(lags) .* SAVE_STRIDE .* KSE.Δt  # lags in time units
ac_keys = (:LS, :int, :ephec)

n_ic = length(DATA["IC_train"])
ac_fom_ic = Vector{Vector{Float64}}(undef, n_ic)
ac_rom_ic = Dict(key => Vector{Matrix{Float64}}(undef, n_ic) for key in ac_keys)

prog = Progress(n_ic)
Threads.@threads for j in 1:n_ic
    ac_fom_ic[j] = tmean_autocorr(DATA["Xtr_all"][1, j], lags)
    for key in ac_keys
        ac_rom_ic[key][j] = analyze_autocorr(rom_ops[key], KSE, DATA["Vr"], DATA["IC_train"][j], DATA["ro"], lags)
    end
    next!(prog)
end

# Mean normalized autocorrelation and mean relative autocorrelation error over all initial conditions
RES["train_AC_lags"] = lags_t
RES["train_AC"] = Dict{Symbol, Array{Float64}}(:fom => mean(ac_fom_ic))
RES["train_AC_ERR"] = Dict{Symbol, Matrix{Float64}}()
for key in ac_keys
    RES["train_AC"][key] = mean(ac_rom_ic[key])
    RES["train_AC_ERR"][key] = reshape(
        [mean(norm(ac_rom_ic[key][j][:, r] - ac_fom_ic[j]) / norm(ac_fom_ic[j]) for j in 1:n_ic) for r in eachindex(DATA["ro"])],
        :, 1)
end
DataFrame(r=DATA["ro"]; (key => vec(RES["train_AC_ERR"][key]) for key in ac_keys)...)

In [ ]:
# Four of the reduced orders (smallest to largest)
ri_plot = round.(Int, range(1, length(DATA["ro"]), length=4))
lout = @layout [grid(2,2) a{0.1w}]
p = plot(layout=lout, size=(1100, 500))
for (plot_id, ri) in enumerate(ri_plot)
    plot!(p[plot_id], lags_t, RES["train_AC"][:LS][:,ri], c=:black, label="opinf")
    plot!(p[plot_id], lags_t, RES["train_AC"][:int][:,ri], c=:orange, label="intrusive")
    plot!(p[plot_id], lags_t, RES["train_AC"][:ephec][:,ri], c=:blue, ls=:dash, lw=2, label="ep-opinf")
    plot!(p[plot_id], lags_t, RES["train_AC"][:fom], c=:green, lw=1, ls=:dash, label="full")
    plot!(p[plot_id], fontfamily="Computer Modern", tickfontsize=13, legend=false)
    plot!(p[plot_id], title="r = $(DATA["ro"][ri])", titlefontsize=15, titlefontfamily="Computer Modern")
    yticks!(p[plot_id], [-0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
end
xlabel!(p[3], "time lag"); xlabel!(p[4], "time lag")
ylabel!(p[1], "avg normalized autocorr."); ylabel!(p[3], "avg normalized autocorr.")
plot!(p[1], left_margin=5mm)
plot!(p[3], bottom_margin=5mm, left_margin=5mm)
plot!(p[4], bottom_margin=5mm)
plot!(p[5], [0 0 0 0], legendfontsize=13, legend=:left, showaxis=false, color=[:black :orange :blue :green],
        legend_column=1, grid=false, fg_color_legend=nothing, label=[" opinf" " intrusive" " ep-opinf" " full"])

In [ ]:
p = plot(DATA["ro"], RES["train_AC_ERR"][:LS], c=:black, marker=(:circle, 3.5, :black), label="OpInf")
plot!(DATA["ro"], RES["train_AC_ERR"][:int], c=:orange, marker=(:cross, 8, :orange), markerstrokewidth=2.5, label="Intrusive")
plot!(DATA["ro"], RES["train_AC_ERR"][:ephec], c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label="EP-OpInf")
plot!(majorgrid=true, legend=:topright)
xlabel!("reduced model dimension " * L" r")
ylabel!("avg normalized autocorrelation error")
xticks!(DATA["ro"])
plot!(left_margin=2mm)
title!("Training", titlefontsize=16)
plot!(fontfamily="Computer Modern", guidefontsize=13, tickfontsize=13,  legendfontsize=13)

### Lyapunov Exponent

We compute the `LE_OPTS.m` largest Lyapunov exponents with the Benettin / Shimada–Nagashima algorithm. A reference trajectory and `m` perturbed trajectories are integrated for `T` time units, the perturbations are re-orthonormalized with a QR decomposition, and the logarithms of the diagonal of `R` are accumulated. This is repeated `N` times after a spin-up of `τ` time units. All models start from the first training initial condition, and the ROMs are perturbed in their reduced coordinates. A ROM of dimension $r$ has only $r$ exponents, so we compute $\min(m, r)$ of them.

The ROMs are integrated with time step `LE_OPTS.Δt = 0.01`. The full-order model is integrated with its training time step $\Delta t = 10^{-3}$, for the following reason. The Crank–Nicolson part of the CNAB scheme is A-stable but not L-stable, so the very stiff high-wavenumber modes of the 512-point discretization ($\lambda \approx -5\cdot 10^{6}$) are damped by only $\approx 4/(|\lambda|\Delta t^2)$ per time unit. With $\Delta t = 0.01$ this is $\approx 0.01$, and these numerical modes show up as spurious exponents $\approx -0.01$ that hide the physical ones ($-0.185, -0.253, \dots$). With $\Delta t = 10^{-3}$ they are pushed below $\approx -0.85$, so only the 10th exponent can be affected. The reduced operators are not stiff, so $\Delta t = 0.01$ is fine for the ROMs.

The demo uses a much shorter spin-up and 10× fewer re-orthonormalizations than the paper ($N = 1000$ instead of $10^4$). The exponents are therefore only converged to roughly two digits. For reference, Edson et al. (2019) report the leading exponents of the $L = 22$ KSE as $0.043, 0.003, 0.002, -0.004, -0.008, -0.185, \dots$, with Kaplan–Yorke dimension ≈ 5.198.

In [ ]:
"""
    lyapunov_exponents(S, u0, opts; rng)

Lyapunov exponents of the system integrated by the CNAB stepper `S` (time step `opts.Δt`),
starting from `u0`. Returns the exponents (length `opts.m`, padded with NaN if the system has
fewer dimensions) and their history after each re-orthonormalization (`opts.m × opts.N`).
"""
function lyapunov_exponents(S::CNABStepper, u0::AbstractVector, opts; rng=MersenneTwister(0))
    n = length(u0)
    m = min(opts.m, n)
    nT = round(Int, opts.T / S.Δt)  # time steps between re-orthonormalizations
    flow(u, k) = cnab_solve(S, u, k; keep=[k])[:, 1]

    λsum = zeros(m)
    λ_hist = fill(NaN, opts.m, opts.N)
    u = flow(u0, round(Int, opts.τ / S.Δt))  # spin-up onto the attractor
    Q = Matrix(qr(randn(rng, n, m)).Q)        # random orthonormal perturbation directions
    for j in 1:opts.N
        unext = flow(u, nT)
        all(isfinite, unext) || return fill(NaN, opts.m), λ_hist  # the model blew up
        W = similar(Q)
        for i in 1:m
            W[:, i] = (flow(u .+ opts.ϵ .* Q[:, i], nT) .- unext) ./ opts.ϵ
        end
        fac = qr(W)
        d = diag(fac.R)
        Q = Matrix(fac.Q) .* sign.(d)'  # make the diagonal of R positive
        λsum .+= log.(abs.(d))
        λ_hist[1:m, j] = λsum ./ (j * opts.T)
        u = unext
    end
    λ = fill(NaN, opts.m)
    λ[1:m] = λsum ./ (opts.N * opts.T)
    return λ, λ_hist
end

# Lyapunov exponents of the ROMs of all orders `ro` (first parameter value only)
function analyze_lyapExp(op, Vr_all, IC, ro, opts)
    LE = Vector{Vector{Float64}}(undef, length(ro))
    LE_hist = Vector{Matrix{Float64}}(undef, length(ro))
    Threads.@threads for j in eachindex(ro)
        r = ro[j]
        Vr = Vr_all[1][:, 1:r]
        S = CNABStepper(op[1].A[1:r, 1:r], UniqueKronecker.extractF(op[1].A2u, r), opts.Δt)
        LE[j], LE_hist[j] = lyapunov_exponents(S, Vr' * IC, opts; rng=MersenneTwister(j))
    end
    return LE, LE_hist
end

# Plot |λ_i(j) - λ_i(N)| over the re-orthonormalization steps j
function plot_le_history(λ_hist; title="")
    m, N = size(λ_hist)
    idx = unique(round.(Int, exp10.(range(0, log10(N - 1), length=200))))
    p = plot()
    for i in 1:m
        all(isnan, λ_hist[i, :]) && continue
        plot!(p, idx, abs.(λ_hist[i, idx] .- λ_hist[i, end]), lw=1.5, label=false)
    end
    C = maximum(filter(isfinite, abs.(λ_hist[:, 1] .- λ_hist[:, end])))
    plot!(p, idx, C ./ idx, c=:black, ls=:dash, lw=1.5, label=L"C/{i}")
    plot!(p, idx, C ./ sqrt.(idx), c=:red, ls=:dash, lw=1.5, label=L"C/\sqrt{i}")
    plot!(p, xscale=:log10, yscale=:log10)
    xlabel!(p, L"i" * "-th reorthonormalization step")
    ylabel!(p, L"|\lambda_i - \lambda_N|")
    plot!(p, fontfamily="Computer Modern", guidefontsize=13, tickfontsize=13, legendfontsize=13, legend=:bottomleft)
    title!(p, title, titlefontsize=14)
    return p
end
;

In [ ]:
le_keys = (:LS, :int, :ephec)
RES["train_LE"] = Dict{Symbol, Any}()
RES["train_LE_all"] = Dict{Symbol, Any}()
;

In [ ]:
# Full-order model, integrated with the time step of the training simulations (see the note above)
LE_OPTS_FOM = merge(LE_OPTS, (Δt=KSE.Δt,))
S_fom = CNABStepper(DATA["op_fom_tr"][1].A, DATA["op_fom_tr"][1].A2u, LE_OPTS_FOM.Δt)
@time RES["train_LE"][:fom], RES["train_LE_all"][:fom] = lyapunov_exponents(S_fom, DATA["IC_train"][1], LE_OPTS_FOM)
println("FOM Lyapunov exponents: ", round.(RES["train_LE"][:fom], digits=3))

In [ ]:
p = plot_le_history(RES["train_LE_all"][:fom]; title="Full Model, $(LE_OPTS.m) Lyapunov Exponent Error History")
savefig(p, "figures/kse_train_le_err_fom.pdf")
p

In [ ]:
for key in le_keys
    RES["train_LE"][key], RES["train_LE_all"][key] = analyze_lyapExp(rom_ops[key], DATA["Vr"], DATA["IC_train"][1], DATA["ro"], LE_OPTS)
end
;

In [ ]:
titles = Dict(:LS => "Standard OpInf", :int => "Intrusive", :ephec => "EP-OpInf")
plot([plot_le_history(RES["train_LE_all"][key][end]; title="$(titles[key]), r = $(DATA["ro"][end])") for key in le_keys]...,
     layout=(1, 3), size=(1500, 420), bottom_margin=12mm, left_margin=8mm)

In [ ]:
# Lyapunov exponents at r = r_max, with the reference values of Edson et al. (2019)
edson_LE = [0.043, 0.003, 0.002, -0.004, -0.008, -0.185, -0.253, -0.296, -0.309, -1.965]
le_df = DataFrame(
    Edson2019=edson_LE[1:LE_OPTS.m],
    fom=sort(RES["train_LE"][:fom], rev=true),
    int=sort(RES["train_LE"][:int][end], rev=true),
    LS=sort(RES["train_LE"][:LS][end], rev=true),
    ephec=sort(RES["train_LE"][:ephec][end], rev=true),
)
le_df

In [ ]:
# Largest Lyapunov exponent vs. reduced dimension, relative to the full-order model
maxLE = Dict(key => [maximum(filter(!isnan, λ); init=NaN) for λ in RES["train_LE"][key]] for key in le_keys)
maxLE_fom = maximum(RES["train_LE"][:fom])

p = plot()
plot!(p, DATA["ro"], maxLE[:int] .- maxLE_fom, c=:orange, marker=(:cross, 8, :orange), markerstrokewidth=2.5, label="Intrusive")
plot!(p, DATA["ro"], maxLE[:LS] .- maxLE_fom, c=:black, marker=(:circle, 3.5, :black), label="OpInf")
plot!(p, DATA["ro"], maxLE[:ephec] .- maxLE_fom, c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label="EP-OpInf")
hline!(p, [0.0], c=:gray, ls=:dot, label=false)
xticks!(p, DATA["ro"])
xlabel!(p, "reduced model dimension " * L" r")
ylabel!(p, L"\lambda_{\max}^{\mathrm{ROM}} - \lambda_{\max}^{\mathrm{FOM}}")
plot!(p, majorgrid=true, fontfamily="Computer Modern", guidefontsize=13, tickfontsize=13, legendfontsize=13, legend=:topright)

### Kaplan–Yorke/Lyapunov Dimension

In [ ]:
function compute_Dky(λs::AbstractArray; sorted::Bool=true)
    λs = filter(!isnan, λs)
    isempty(λs) && return NaN
    if !sorted
        λs = sort(λs, rev=true)
    end

    # Find the index where the sum of λ becomes negative
    j = nothing
    tmp = 0
    for (i,λ) in enumerate(λs)
        tmp += λ
        if tmp < 0
            j = i
            break
        end
    end

    if isnothing(j)
        return NaN  # the sum never becomes negative: more exponents would be needed
    elseif j == 1
        return 0.0
    else
        return (j - 1) + sum(λs[1:j-1]) / abs(λs[j])
    end
end

analyze_Dky(LEs; sorted=false) = [compute_Dky(λ; sorted=sorted) for λ in LEs]
;

In [ ]:
RES["train_DKY"] = Dict{Symbol, Any}()
RES["train_DKY"][:fom] = compute_Dky(RES["train_LE"][:fom]; sorted=false)
for key in le_keys
    RES["train_DKY"][key] = analyze_Dky(RES["train_LE"][key])
end
;

In [ ]:
dky_df = DataFrame(
    order=vec(DATA["ro"]),
    int=RES["train_DKY"][:int],
    LS=RES["train_DKY"][:LS],
    ephec=RES["train_DKY"][:ephec],
)
println("FOM Dky:     ", RES["train_DKY"][:fom])
println("Ref FOM Dky: ", round(compute_Dky(edson_LE), digits=3), " (Edson et al. 2019)")
dky_df

In [ ]:
plot(DATA["ro"], abs.(RES["train_DKY"][:LS] .- RES["train_DKY"][:fom]), c=:black, marker=(:circle, 3.5, :black), label="opinf")
plot!(DATA["ro"], abs.(RES["train_DKY"][:int] .- RES["train_DKY"][:fom]), c=:orange, marker=(:cross, 8, :orange), label="intrusive")
plot!(DATA["ro"], abs.(RES["train_DKY"][:ephec] .- RES["train_DKY"][:fom]), c=:blue, markerstrokecolor=:blue, marker=(:rect, 3), ls=:dash, lw=2, label="ephec-opinf")
plot!(majorgrid=true, legend=:topright)
xlabel!("reduced model dimension " * L" r")
ylabel!("error of Kaplan-Yorke dimension")
xticks!(DATA["ro"])
plot!(fontfamily="Computer Modern", guidefontsize=13, tickfontsize=13,  legendfontsize=13)

### Flow Field Comparison

In [ ]:
i = 1
r = DATA["ro"][end]
ic = DATA["IC_train"][i]
Vr_plot = DATA["Vr"][1][:, 1:r]

tspan = KSE.tspan
X_fom = DATA["Xtr_all"][1, i]
X_int = simulate_rom(KSE, OPS["op_int"][1], r, Vr_plot' * ic)
X_LS = simulate_rom(KSE, OPS["op_LS"][1], r, Vr_plot' * ic)
X_ephec = simulate_rom(KSE, OPS["op_ephec"][1], r, Vr_plot' * ic)

# Snapshots every EVAL_STRIDE time steps
t_plot = tspan[1:EVAL_STRIDE:end]
U_fom = X_fom[:, 1:(EVAL_STRIDE ÷ SAVE_STRIDE):end]
U_int = Vr_plot * X_int[:, 1:EVAL_STRIDE:end]
U_LS = Vr_plot * X_LS[:, 1:EVAL_STRIDE:end]
U_ephec = Vr_plot * X_ephec[:, 1:EVAL_STRIDE:end]
;

In [ ]:
lout = @layout [grid(4,2)]
p_fom = plot(
    contourf(t_plot, KSE.xspan, U_fom, lw=0, color=:inferno),
    colorbar_ticks=(-3:3), yticks=(0:5:20), ylims=(0,22),
    left_margin=15mm,
    ylabel=L"\textbf{Full}" * "\n" * L"\omega",
    xformatter=_->"",
)
pblank = plot(
    axis=false,legend=false,grid=false, background_color_inside=:transparent,
)
p_int = plot(
    contourf(t_plot, KSE.xspan, U_int, lw=0, color=:inferno),
    colorbar_ticks=(-3:3), yticks=(0:5:20), ylims=(0,22),
    left_margin=15mm,
    ylabel=L"\textbf{Intrusive}" * "\n" * L"\omega",
    xformatter=_->"",
)
p_int_err = plot(
    contourf(t_plot, KSE.xspan, abs.(U_fom .- U_int), lw=0, color=:roma),
    yticks=(0:5:20), ylims=(0,22), colorbar_ticks=(0:0.5:5), clim=(0,5),
    xformatter=_->"", yformatter=_->"",
)
p_LS = plot(
    contourf(t_plot, KSE.xspan, U_LS, lw=0, color=:inferno),
    colorbar_ticks=(-3:3), yticks=(0:5:20), ylims=(0,22),
    left_margin=15mm,
    ylabel=L"\textbf{OpInf}" * "\n" * L"\omega",
    xformatter=_->"",
)
p_LS_err = plot(
    contourf(t_plot, KSE.xspan, abs.(U_fom .- U_LS), lw=0, color=:roma),
    yticks=(0:5:20), ylims=(0,22),  colorbar_ticks=(0:0.5:5), clim=(0,5),
    xformatter=_->"", yformatter=_->"",
)
p_ephec = plot(
    contourf(t_plot, KSE.xspan, U_ephec, lw=0, color=:inferno),
    colorbar_ticks=(-3:3), yticks=(0:5:20), ylims=(0,22),
    bottom_margin=15mm, left_margin=15mm,
    ylabel=L"\textbf{EP}\rm{-}\textbf{OpInf}" * "\n" * L"\omega", xlabel=L"t" * "\n" * L"\textbf{Predicted}~" * L"x(\omega,t)",
)
p_ephec_err = plot(
    contourf(t_plot, KSE.xspan, abs.(U_fom .- U_ephec), lw=0, color=:roma),
    yticks=(0:5:20), ylims=(0,22), colorbar_ticks=(0:0.5:5), clim=(0,5),
    bottom_margin=15mm,
    xlabel=L"t" * "\n" * L"\textbf{Error}",
    yformatter=_->"",
)

p = plot(
    p_fom, pblank,
    p_int, p_int_err,
    p_LS, p_LS_err,
    p_ephec, p_ephec_err,
    fontfamily="Computer Modern", layout=lout,
    size=(2000, 1080),
    guidefontsize=25, tickfontsize=17, plot_titlefontsize=30,
    plot_title="Predicted Flow Fields and Errors of Training (r = $(r))",
)

### Save the results

In [ ]:
save(resultfile, RES)

## Remarks

- The energy-preserving constraint residual of the EPHEC operators is at round-off level, as for the intrusive and full-order operators. Standard OpInf violates the constraint, and the violation grows with $r$. EPSIC (each constraint relaxed to $\pm\epsilon$) and EPP (penalty) reduce it without enforcing it exactly. In the demo run, EPSIC reduces it by about 6× and EPP by about 750× at $r_{\max}$.
- Because the KSE is chaotic, the state errors over $t \in [0, 300]$ are large for every model. The ROMs should instead be judged by how well they reproduce the statistics of the attractor (autocorrelation, Lyapunov spectrum, Kaplan–Yorke dimension).
- The full-order Lyapunov spectrum agrees well with the reference values of Edson et al. (Kaplan–Yorke dimension ≈ 5.2). The larger ROMs reproduce the leading exponent and the strongly negative exponents. In the demo they show one fewer near-zero exponent than the FOM, so their Kaplan–Yorke dimension is ≈ 4.2. A Kaplan–Yorke dimension of 0 means that the computed leading exponent is negative, i.e. the ROM settled on a non-chaotic attractor for this initial condition.
- The demo uses fewer training snapshots (`DS = 1000`), a smaller $r_{\max}$ and shorter Lyapunov-exponent runs than the paper, so its numbers differ from the published ones. Set `PAPER_SCALE = true` for the full experiment.
- Notebook `07_KSE_EP-OpInf_test.ipynb` tests these ROMs on new initial conditions.